# CISEI Scenario Builder — somente `nodes.csv` e `scenario.toml`

Este notebook produz exclusivamente os dois artefatos de autoria do projeto:

- `nodes.csv`, preparado **antes da inicialização do agente e da primeira mensagem ao LLM**;
- `scenario.toml`, construído na conversa e gravado somente após validação e confirmação explícita.

## Fluxo para um projeto novo ou existente

1. Informar o nome do projeto. Se ele ainda não existir, o notebook cria `projects/<nome>/data/`.
2. Carregar os arquivos-fonte na pasta `data/`, usando o controle de upload abaixo ou o navegador de arquivos do Jupyter.
3. Clicar em **Preparar nodes.csv**. O compilador decide se uma fonte já basta ou se precisa combinar fontes. Se faltarem dados, mostra o diagnóstico e permite carregar mais arquivos.
4. Só depois que `nodes.csv` estiver validado, executar as células seguintes para inicializar o LLM e conversar.
5. Conversar livremente e usar os botões de confirmação e geração após validação.

**Importante:** não execute as células do agente antes de preparar `nodes.csv`. Se não houver arquivos suficientes, o notebook interrompe a inicialização com uma orientação clara. Ao trabalhar num projeto novo, use primeiro a célula de preparação, faça o upload e só então continue.

O notebook não executa planejamento, não chama MCP nem Planning Service, não calcula métricas, não resolve ou avalia enlaces, não plota, não renderiza mapas, não gera `features.json` nem `result.json`. Os arquivos carregados em `data/` são fontes de entrada; os únicos artefatos de saída produzidos pelo fluxo são `nodes.csv` e `scenario.toml`.

**Validação pós-LLM:** antes de gravar `scenario.toml`, a etapa 5 executa uma segunda checagem local do texto gerado contra `nodes.csv`. A validação de `PlanningScenario.validate()` no motor não é executada neste notebook.

**Validação SDK:** na geração final, um identificador exclusivo é registrado no serviço para executar `PlanningScenario.validate()`. Não usa o identificador do planejamento ativo.


**O que esta célula faz:** Instala, se necessário, o SDK local e importa as bibliotecas usadas pelas demais etapas.


In [ ]:
# Dependências do Scenario Builder e da interface conversacional.
from pathlib import Path
import importlib.util
import subprocess
import sys
import json
import re
import tomllib
import ipywidgets as widgets
from IPython.display import display

PROJECT_CODE_ROOT = Path.home() / "work" / "planapp-mcp"
SDK_WHEEL = PROJECT_CODE_ROOT / "vendor" / "cisei_planning_sdk-0.1.18-py3-none-any.whl"

if importlib.util.find_spec("cisei_planning_sdk") is None:
    if not SDK_WHEEL.is_file():
        raise FileNotFoundError(f"SDK não encontrado: {SDK_WHEEL}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", str(SDK_WHEEL)],
        check=True,
    )


## 1. Criar/selecionar o projeto e preparar as fontes

Informe o nome do projeto, mesmo que ainda não exista. O notebook cria o diretório do projeto e `data/`. Em seguida, carregue ali os arquivos que poderão servir como fontes para `nodes.csv` e execute a preparação. O agente conversacional só será inicializado depois que o CSV estiver válido.


**Esta etapa cria diretórios, recebe fontes de entrada e prepara somente `nodes.csv`.** Se ainda não houver arquivos suficientes, ela permanece na etapa de preparação e explica como carregar mais fontes; não inicia o LLM nem cria outros artefatos de saída.


In [ ]:
WORKSPACE_ROOT = Path("/home/jovyan/work/cisei_workspace")
PROJECTS_ROOT = WORKSPACE_ROOT / "projects"

PROJECT_NAME = input(
    "Nome do projeto (ex.: copel_lte450_minor): "
).strip()

if not PROJECT_NAME:
    raise ValueError("O nome do projeto não pode ser vazio.")

if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", PROJECT_NAME) or ".." in PROJECT_NAME:
    raise ValueError(
        "Nome inválido. Use letras, números, ponto, hífen e underscore; "
        "o nome deve começar com letra ou número e não conter '..'."
    )

PROJECT_ROOT = PROJECTS_ROOT / PROJECT_NAME
DATA_ROOT = PROJECT_ROOT / "data"
SCENARIO_PATH = PROJECT_ROOT / "scenario.toml"
NODES_PATH = PROJECT_ROOT / "nodes.csv"
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# O compilador roda antes de qualquer inicialização do agente/LLM.
if str(PROJECT_CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_CODE_ROOT))
COMPILER_MODULE = PROJECT_CODE_ROOT / "nodes_compiler.py"
if not COMPILER_MODULE.is_file():
    raise FileNotFoundError(
        f"nodes_compiler.py não encontrado: {COMPILER_MODULE}. "
        "Instale/copie o módulo antes de preparar nodes.csv."
    )

from nodes_compiler import NodesCompiler

# O compilador lê nodes_config.toml do projeto automaticamente, quando necessário.
NODES_COMPILER = NodesCompiler(PROJECT_ROOT)
NODES_READY = False
NODES_PREPARATION_REPORT = None

node_preparation_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc", padding="10px", width="100%")
)
source_upload = widgets.FileUpload(
    accept="",
    multiple=True,
    description="Selecionar arquivos",
    layout=widgets.Layout(width="100%"),
)
replace_uploads_checkbox = widgets.Checkbox(
    value=False,
    description="Permitir substituir arquivos de mesmo nome em data/",
    indent=False,
)
prepare_nodes_button = widgets.Button(
    description="Preparar nodes.csv",
    button_style="success",
    icon="check",
)
save_uploads_button = widgets.Button(
    description="Salvar uploads em data/ e preparar",
    button_style="primary",
    icon="upload",
)


def _uploaded_file_entries(widget_value):
    """Normaliza FileUpload do ipywidgets 7 (dict) e 8 (tupla/lista)."""
    if isinstance(widget_value, dict):
        entries = []
        for filename, metadata in widget_value.items():
            item = dict(metadata)
            item.setdefault("name", filename)
            entries.append(item)
        return entries
    if isinstance(widget_value, (tuple, list)):
        return [dict(item) for item in widget_value]
    return []


def _prepare_nodes_csv():
    """Prepara ou valida nodes.csv; não inicializa o agente."""
    global NODES_READY, NODES_PREPARATION_REPORT, NODES_PATH
    NODES_READY = False
    NODES_PREPARATION_REPORT = None

    if not NODES_PATH.is_file() and not NODES_COMPILER.inspect_sources():
        print("Projeto criado:", PROJECT_ROOT)
        print("Pasta de entrada criada:", DATA_ROOT)
        print("Ainda não há arquivos em data/. Carregue os arquivos e tente novamente.")
        print("É possível usar o seletor abaixo ou enviar os arquivos pela interface de arquivos do Jupyter para:")
        print(DATA_ROOT)
        return False

    try:
        NODES_PATH = NODES_COMPILER.prepare_output(overwrite_existing=False)
        if NODES_COMPILER.last_report is None:
            raise RuntimeError("O compilador terminou sem produzir um relatório.")
        NODES_PREPARATION_REPORT = NODES_COMPILER.last_report.as_dict()
        NODES_READY = NODES_PATH.is_file()
        print("=== PREPARAÇÃO DE nodes.csv ===")
        print(json.dumps(NODES_PREPARATION_REPORT, indent=2, ensure_ascii=False))
        print("Arquivo de nós:", NODES_PATH)
        print("Nós validados:", NODES_COMPILER.last_report.rows_written)
        print("Referência que será usada no scenario.toml: instances.source = 'nodes.csv'")
        print("Pronto para inicializar o agente:", NODES_READY)
        return NODES_READY
    except Exception as exc:
        print("nodes.csv ainda não foi preparado.")
        print(f"Diagnóstico: {type(exc).__name__}: {exc}")
        print("Carregue outras fontes em data/ ou corrija os arquivos indicados; depois clique em Preparar nodes.csv novamente.")
        print("Fontes encontradas:")
        paths = NODES_COMPILER.inspect_sources()
        if not paths:
            print("(nenhuma)")
        else:
            for path in paths:
                print(" -", path.relative_to(DATA_ROOT))
        return False


def _prepare_nodes_clicked(_):
    with node_preparation_output:
        node_preparation_output.clear_output(wait=True)
        _prepare_nodes_csv()


def _save_uploads_and_prepare_clicked(_):
    with node_preparation_output:
        node_preparation_output.clear_output(wait=True)
        try:
            entries = _uploaded_file_entries(source_upload.value)
            if not entries:
                print("Nenhum arquivo novo foi selecionado. Vou tentar preparar usando os arquivos que já existem em data/.")
                _prepare_nodes_csv()
                return

            # Valida todos os nomes/conflitos antes de gravar qualquer arquivo.
            pending = []
            for item in entries:
                raw_name = str(item.get("name") or "").strip()
                filename = Path(raw_name).name
                if not filename or filename in {".", ".."} or filename != raw_name or "\\" in raw_name:
                    raise ValueError(f"Nome de arquivo inválido para upload: {raw_name!r}")
                content = item.get("content", b"")
                if hasattr(content, "tobytes"):
                    content = content.tobytes()
                elif not isinstance(content, bytes):
                    content = bytes(content)
                target = DATA_ROOT / filename
                if target.exists():
                    existing = target.read_bytes()
                    if existing == content:
                        print(f"Já existe com conteúdo idêntico; mantido: {filename}")
                        continue
                    if not replace_uploads_checkbox.value:
                        raise FileExistsError(
                            f"Já existe um arquivo diferente chamado {filename!r} em data/. "
                            "Habilite a substituição explicitamente ou renomeie o arquivo. Nenhum upload novo foi gravado."
                        )
                pending.append((target, content))

            for target, content in pending:
                target.write_bytes(content)
                print(f"Fonte salva: {target.relative_to(PROJECT_ROOT)} ({len(content)} bytes)")

            _prepare_nodes_csv()
        except Exception as exc:
            print(f"Falha ao salvar/preparar fontes: {type(exc).__name__}: {exc}")
            print("As fontes existentes foram preservadas. Corrija o conflito ou selecione a substituição explicitamente e tente novamente.")

prepare_nodes_button.on_click(_prepare_nodes_clicked)
save_uploads_button.on_click(_save_uploads_and_prepare_clicked)

print("Projeto:", PROJECT_NAME)
print("Diretório:", PROJECT_ROOT)
print("Pasta para arquivos de entrada:", DATA_ROOT)
print("scenario.toml:", "já existe" if SCENARIO_PATH.is_file() else "será gerado ao final")
print("Carregue os arquivos de entrada abaixo. O LLM não será inicializado até nodes.csv estar validado.")

display(widgets.VBox([
    widgets.HTML("<h4>Arquivos de entrada para nodes.csv</h4>"),
    widgets.HTML(f"<p>Projeto: <code>{PROJECT_ROOT}</code><br>Pasta de fontes: <code>{DATA_ROOT}</code></p>"),
    source_upload,
    replace_uploads_checkbox,
    widgets.HBox([save_uploads_button, prepare_nodes_button]),
    node_preparation_output,
]))

# Se o projeto já tinha nodes.csv ou fontes em data/, tenta preparar automaticamente.
# Ausência de fontes é um estado normal: o usuário poderá fazer upload pela UI acima.
with node_preparation_output:
    _prepare_nodes_csv()


### Configuração de conexão com o SDK

Utiliza o endereço do `network-service` já testado no Jupyter. A conexão registra o cliente; a validação usa um identificador de cenário exclusivo.

In [ ]:
# Endereço confirmado no Jupyter para o network-service.
# Pode ser alterado explicitamente aqui se a rede Docker mudar.
SDK_BASE_URL = "http://cisei-planning-stack-network-service-1:8080"
SDK_WORKSPACE = "/home/jovyan/work/cisei_workspace"


def criar_cliente_validacao_sdk():
    from cisei_planning_sdk.client import PlanningClient
    return PlanningClient(
        base_url=SDK_BASE_URL,
        workspace=SDK_WORKSPACE,
        auto_register=True,
        timeout=120.0,
    )


# Associação LOCAL do projeto ao cliente, sem chamar open_project().
# open_project() redefiniria o cenário com o ID original no servidor.
def associar_projeto_validacao(client, pasta_projeto):
    from cisei_planning_sdk.project import PlanningProject
    import inspect

    pasta = Path(pasta_projeto).resolve()
    # PlanningProject representa o projeto local; nenhuma chamada HTTP aqui.
    parametros = inspect.signature(PlanningProject).parameters
    if "path" in parametros:
        projeto_sdk = PlanningProject(path=pasta)
    elif "root" in parametros:
        projeto_sdk = PlanningProject(root=pasta)
    else:
        raise RuntimeError(
            "Assinatura de PlanningProject não reconhecida: "
            + str(inspect.signature(PlanningProject))
        )
    client.project = projeto_sdk
    return projeto_sdk


## Validação independente de um projeto existente (SDK)

Esta seção funciona **sem inicializar o LLM** e sem gerar novamente `nodes.csv` ou `scenario.toml`. Informe o nome de qualquer projeto em `cisei_workspace/projects/` e clique em **VALIDAR CENÁRIO EXISTENTE**. O notebook lê os dois arquivos, verifica a associação de perfis e invoca `PlanningScenario.validate()` por meio do SDK. Não executa planejamento, métricas ou mapas.

**Atenção:** a chamada `define_scenario()` registra um identificador novo no `network-service` (sem substituir o cenário ativo); o registro não é removido automaticamente. A validação depende de conexão com o serviço.

In [ ]:
# Independente do agente, da confirmação e da geração dos artefatos.
import uuid
import json
import tomllib
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display


def validar_projeto_existente(nome_projeto: str) -> dict:
    """Lê nodes.csv e scenario.toml já existentes; não os altera."""
    nome = str(nome_projeto).strip()
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", nome) or ".." in nome:
        raise ValueError("Nome de projeto inválido.")
    projeto = (PROJECTS_ROOT / nome).resolve()
    if projeto.parent != PROJECTS_ROOT.resolve() or not projeto.is_dir():
        raise FileNotFoundError(f"Projeto não encontrado: {projeto}")
    csv_path = projeto / "nodes.csv"
    toml_path = projeto / "scenario.toml"
    for path in (csv_path, toml_path):
        if not path.is_file():
            raise FileNotFoundError(f"Arquivo obrigatório não encontrado: {path}")

    # Validação do CSV existente, sem prepare_output ou recompilação.
    from nodes_compiler import NodesCompiler
    rows = NodesCompiler(projeto).read_csv(csv_path)
    scenario_data = tomllib.loads(toml_path.read_text(encoding="utf-8"))
    instances = scenario_data.get("instances", {})
    if not isinstance(instances, dict):
        raise ValueError("Tabela [instances] ausente ou inválida.")
    expected = {"source": "nodes.csv", "format": "csv", "profile_column": "device_profile"}
    for key, value in expected.items():
        if instances.get(key) != value:
            raise ValueError(f"instances.{key}: esperado {value!r}, recebido {instances.get(key)!r}")
    devices = scenario_data.get("devices", {})
    if not isinstance(devices, dict):
        raise ValueError("Tabela [devices] ausente ou inválida.")
    profiles = sorted({str(row.get("device_profile", "")).strip() for row in rows})
    missing = [profile for profile in profiles if not profile or profile not in devices]
    if missing:
        raise ValueError("Perfis do nodes.csv sem definição em [devices]: " + ", ".join(repr(p) for p in missing))

    # O SDK carrega o TOML a partir do caminho real, preservando a resolução
    # relativa de instances.source = nodes.csv. ID novo evita clobber do ativo.
    try:
        from cisei_planning_sdk.client import PlanningClient
    except ImportError:
        from cisei_planning_sdk.client import PlannerClient as PlanningClient
    validation_id = f"validation_{nome}_{uuid.uuid4().hex}"
    client = criar_cliente_validacao_sdk()
    associar_projeto_validacao(client, projeto)
    session = client.define_scenario(name=validation_id, scenario=toml_path, planner="cell")
    errors = list(session.validation_errors or [])
    return {
        "projeto": nome,
        "nodes": len(rows),
        "perfis": profiles,
        "toml": str(toml_path),
        "validador": "SDK / PlanningScenario.validate()",
        "sdk_ok": bool(session.ok) and not errors,
        "validation_errors": errors,
        "summary": session.summary,
        "validation_id": validation_id,
        "observacao": "ID de validação registrado no servidor; não removido automaticamente.",
    }


projeto_para_validar = widgets.Text(value=PROJECT_NAME, description="Projeto:",
    layout=widgets.Layout(width="70%"))
botao_validar_existente = widgets.Button(description="VALIDAR CENÁRIO EXISTENTE",
    button_style="info", icon="check-circle")
saida_validacao_existente = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc", padding="10px", width="100%"))


def _clicar_validar_existente(_):
    with saida_validacao_existente:
        saida_validacao_existente.clear_output(wait=True)
        try:
            report = validar_projeto_existente(projeto_para_validar.value)
            print("RESULTADO:", "APROVADO PELO SDK" if report["sdk_ok"] else "REPROVADO PELO SDK")
            print(json.dumps(report, indent=2, ensure_ascii=False, default=str))
        except Exception as exc:
            print(f"VALIDAÇÃO NÃO CONCLUÍDA: {type(exc).__name__}: {exc}")
            print("Nenhum arquivo de autoria foi alterado.")


botao_validar_existente.on_click(_clicar_validar_existente)
display(widgets.VBox([widgets.HTML("<h3>Validar arquivos já existentes</h3>"),
    projeto_para_validar, botao_validar_existente, saida_validacao_existente]))

## 2. Inicializar o agente conversacional

Só execute esta seção depois que a célula 1 indicar que `nodes.csv` está pronto. Para um projeto novo, primeiro carregue os arquivos na pasta `data/` e use o botão **Salvar uploads em data/ e preparar** ou **Preparar nodes.csv**. Esta seção não executa planejamento nem se conecta a serviços de execução.


**Pré-condição:** `NODES_READY` deve ser `True` e o arquivo `nodes.csv` deve existir e estar validado. Caso contrário, a inicialização do agente será bloqueada sem enviar mensagens ao LLM.


In [ ]:
if not globals().get("NODES_READY", False) or not NODES_PATH.is_file():
    raise RuntimeError(
        "O agente NÃO foi inicializado porque nodes.csv ainda não está pronto. "
        "Volte à célula de preparação do projeto, carregue os arquivos em "
        f"{DATA_ROOT}, clique em 'Salvar uploads em data/ e preparar' ou "
        "'Preparar nodes.csv' e execute esta célula novamente."
    )

# Revalidação estrutural imediatamente antes da primeira chamada ao agente/LLM.
# Isso detecta alterações no arquivo após a preparação inicial.
NODES_VALIDATED_ROWS = NODES_COMPILER.read_csv(NODES_PATH)
NODES_READY = True

if str(PROJECT_CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_CODE_ROOT))

PLUGIN_ROOT = PROJECT_CODE_ROOT / "cisei-planning-engineering"

from scenario_agent_openai import create_scenario_agent_openai
from scenario_domain.providers import CISEIPluginDomainContextProvider

DOMAIN_CONTEXT_SECTIONS = (
    "scenario_format",
    "assistant_workflow",
    "design_gaps_and_decisions",
)
provider = CISEIPluginDomainContextProvider(PLUGIN_ROOT)
agent = create_scenario_agent_openai(
    PROJECT_ROOT,
    domain_context_provider=provider,
    domain_context_sections=DOMAIN_CONTEXT_SECTIONS,
)

# Esta vinculação é configurada antes de qualquer chamada a agent.chat().
NODES_INSTANCE_BINDING = {
    "source": "nodes.csv",
    "format": "csv",
    "profile_column": "device_profile",
}
conversation_session = agent.scenario_agent.conversation.session
if hasattr(conversation_session, "update") and callable(conversation_session.update):
    conversation_session.update({"instances": NODES_INSTANCE_BINDING})
else:
    conversation_session.builder.update({"instances": NODES_INSTANCE_BINDING})

current_instances = conversation_session.builder.spec.get("instances", {})
for key, expected in NODES_INSTANCE_BINDING.items():
    if current_instances.get(key) != expected:
        raise RuntimeError(
            f"Configuração inicial incorreta: instances.{key} deveria ser {expected!r}."
        )

print("ScenarioAgentOpenAI inicializado após preparar nodes.csv.")
print("Projeto:", PROJECT_NAME)
print("nodes.csv:", NODES_PATH, f"({len(NODES_VALIDATED_ROWS)} nós)")
print("instances:", json.dumps(NODES_INSTANCE_BINDING, ensure_ascii=False))
print("Contexto do plugin:", ", ".join(DOMAIN_CONTEXT_SECTIONS))


## 3. Conversar com o agente

Envie mensagens para fornecer e revisar os parâmetros técnicos do cenário. `nodes.csv` já existe, e `instances.source` já aponta para ele. O agente deve pedir os dados técnicos ausentes, sem usar exemplos como defaults.


**Esta célula fornece somente a interface conversacional e a consulta textual do estado de validação do cenário. Não contém visualizações de rede ou mapas.**


In [ ]:
def obter_estado():
    return agent.scenario_agent.conversation_state()

def mostrar_estado():
    estado = obter_estado()
    print(json.dumps(estado, indent=2, ensure_ascii=False, default=str))
    return estado

def conversar(mensagem: str):
    mensagem = str(mensagem).strip()
    if not mensagem:
        return None
    return agent.chat(mensagem)

entrada = widgets.Textarea(
    placeholder="Digite sua mensagem para o ScenarioAgentOpenAI...",
    layout=widgets.Layout(width="100%", min_height="100px"),
)
botao_enviar = widgets.Button(description="ENVIAR", button_style="primary")
botao_limpar = widgets.Button(description="LIMPAR")
botoes_mensagem = widgets.HBox([botao_enviar, botao_limpar],
    layout=widgets.Layout(align_items="center", gap="8px"))

# Saída dedicada: mostra SOMENTE a resposta à mensagem mais recente.
agent_reply_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #8aa4c8", padding="10px",
                          width="100%", min_height="70px")
)
# Histórico cumulativo: mantém todas as mensagens e respostas abaixo da saída dedicada.
conversation_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc", padding="10px",
                          width="100%", max_height="500px", overflow="auto")
)
state_output = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc", padding="10px",
                          width="100%", max_height="350px", overflow="auto")
)

def _texto_resposta(resposta):
    if isinstance(resposta, dict):
        acao = resposta.get("action")
        if acao == "ask_user" and resposta.get("question"):
            return str(resposta["question"])
        if acao == "update_scenario":
            return "Informações recebidas e encaminhadas para validação. Consulte o estado atual do cenário abaixo."
        if acao == "confirm":
            return "Confirmação processada. Consulte o estado do cenário abaixo."
        if acao == "finalize":
            return "Finalização processada. Verifique o arquivo scenario.toml na seção de artefatos."
    if isinstance(resposta, (dict, list)):
        return json.dumps(resposta, indent=2, ensure_ascii=False, default=str)
    return str(resposta) if resposta is not None else "O agente não retornou conteúdo."

def enviar_mensagem(_):
    mensagem = entrada.value.strip()
    if not mensagem:
        return

    # A saída superior é substituída a cada envio; o histórico inferior é cumulativo.
    agent_reply_output.clear_output(wait=True)
    with agent_reply_output:
        print("Retorno do agente IA")
        print("=" * 48)
        try:
            resposta = conversar(mensagem)
            resposta_texto = _texto_resposta(resposta)
            print(resposta_texto)
        except Exception as exc:
            resposta_texto = f"ERRO: {type(exc).__name__}: {exc}"
            print(resposta_texto)

    with conversation_output:
        print("=" * 72)
        print("USUÁRIO")
        print("=" * 72)
        print(mensagem)
        print("\n" + "=" * 72)
        print("RETORNO DO AGENTE IA")
        print("=" * 72)
        print(resposta_texto)
        print()

    entrada.value = ""
    with state_output:
        state_output.clear_output(wait=True)
        try:
            mostrar_estado()
        except Exception as exc:
            print(f"Não foi possível consultar o estado: {type(exc).__name__}: {exc}")

def limpar_saida(_):
    # Limpa as áreas visuais, sem apagar o estado interno da sessão/agente.
    agent_reply_output.clear_output()
    conversation_output.clear_output()
    state_output.clear_output()

botao_enviar.on_click(enviar_mensagem)
botao_limpar.on_click(limpar_saida)

display(widgets.VBox([
    entrada,
    botoes_mensagem,
    widgets.HTML("<h3>Retorno do agente IA</h3>"),
    agent_reply_output,
    widgets.HTML("<h3>Histórico da conversa</h3>"),
    conversation_output,
    widgets.HTML("<h3>Estado atual do cenário</h3>"),
    state_output,
]))

# Ações do cliente: nenhuma confirmação ou gravação é automática.
confirmar_ui = widgets.Button(description="CONFIRMAR CENÁRIO", button_style="warning", icon="check")
gerar_ui = widgets.Button(description="GERAR scenario.toml", button_style="success", icon="save")
substituir_ui = widgets.Checkbox(value=False, description="Permitir substituir scenario.toml existente", indent=False)
acoes_output = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="10px", width="100%"))

def _confirmar_ui(_):
    with acoes_output:
        acoes_output.clear_output(wait=True)
        try:
            estado = obter_estado()
            if not (estado.get("valid") and estado.get("complete")):
                print("Cenário ainda não está válido e completo. Continue a conversa.")
                return
            resultado = agent.chat("Confirmo o cenário.")
            print(_texto_resposta(resultado))
            with conversation_output:
                print("=" * 72)
                print("USUÁRIO\nConfirmo o cenário.\nRETORNO DO AGENTE IA")
                print(_texto_resposta(resultado))
            with state_output:
                state_output.clear_output(wait=True)
                mostrar_estado()
        except Exception as exc:
            print(f"Confirmação bloqueada: {type(exc).__name__}: {exc}")

def _gerar_ui(_):
    with acoes_output:
        acoes_output.clear_output(wait=True)
        try:
            estado = obter_estado()
            if not all(estado.get(k) is True for k in ("valid", "complete", "confirmed")):
                print("Cenário ainda não está válido, completo e confirmado.")
                return
            # Função definida na seção 5; evita gravar sem executar a célula de validação.
            if "gerar_e_salvar_scenario" not in globals():
                print("Execute primeiro a célula 'Gerar scenario.toml' para habilitar a validação final.")
                return
            gerar_e_salvar_scenario(overwrite=substituir_ui.value)
        except Exception as exc:
            print(f"Geração bloqueada: {type(exc).__name__}: {exc}")

confirmar_ui.on_click(_confirmar_ui)
gerar_ui.on_click(_gerar_ui)
display(widgets.VBox([
    widgets.HTML("<h3>Confirmar e gerar scenario.toml</h3>"),
    widgets.HTML("<p>Confirme apenas após o cenário estar válido e completo. A gravação exige confirmação explícita.</p>"),
    widgets.HBox([confirmar_ui, gerar_ui]),
    substituir_ui,
    acoes_output,
]))


**Relatório textual da preparação de `nodes.csv`.** Mostra a fonte ou combinação escolhida e os diagnósticos. Não abre mapas, não gera gráficos e não cria arquivos de relatório.


In [ ]:
if not NODES_READY or NODES_PREPARATION_REPORT is None:
    print("nodes.csv ainda não está pronto.")
    print("Carregue fontes em:", DATA_ROOT)
    print("Depois use o botão de preparação na célula do projeto antes de iniciar a conversa.")
else:
    print("=== PREPARAÇÃO DE nodes.csv ===")
    print(json.dumps(NODES_PREPARATION_REPORT, indent=2, ensure_ascii=False))
    print(f"Arquivo de nós: {NODES_PATH}")
    print(f"Existe e está validado: {NODES_PATH.is_file()}")
    print("Entrada que será referenciada pelo TOML: nodes.csv")


## 4. Validar e confirmar o cenário

Revise as informações fornecidas na conversa. A confirmação só é enviada depois de habilitar explicitamente a opção na célula abaixo.


**O que esta célula faz:** Consulta o estado do cenário e verifica se ele está válido e completo para confirmação.


In [ ]:
def validar_perfis_nodes_scenario(scenario_data=None):
    """Diagnostica perfis do CSV sem renomear/mutar dados nem bloquear por discrepância confirmada.

    A validação estrutural obrigatória continua sendo ScenarioBuilder.validate().
    Este diagnóstico apenas registra a divergência entre os nomes do CSV e [devices].
    """
    rows = NODES_COMPILER.read_csv(NODES_PATH)  # leitura apenas; nunca recompila nodes.csv
    if scenario_data is None:
        scenario_data = conversation_session.builder.spec
    if not isinstance(scenario_data, dict):
        raise ValueError("A especificação do cenário não é um dicionário.")

    instances = scenario_data.get("instances", {})
    if not isinstance(instances, dict) or instances.get("source") != "nodes.csv":
        raise ValueError(
            "A validação nível 1 exige instances.source = 'nodes.csv'; "
            f"encontrado: {instances.get('source') if isinstance(instances, dict) else instances!r}."
        )
    profile_column = instances.get("profile_column", "device_profile")
    if profile_column != "device_profile":
        raise ValueError(
            "O CSV preparado usa a coluna 'device_profile'; "
            f"a especificação aponta para {profile_column!r}."
        )

    profiles_csv = sorted({str(row.get("device_profile", "")).strip()
                           for row in rows if str(row.get("device_profile", "")).strip()})
    devices = scenario_data.get("devices", {})
    if not isinstance(devices, dict):
        raise ValueError("A seção [devices] precisa ser uma tabela de perfis.")
    missing_profiles = [profile for profile in profiles_csv if profile not in devices]
    return {
        "rows": len(rows),
        "profiles_csv": profiles_csv,
        "devices_defined": sorted(devices),
        "profiles_without_device_definition": missing_profiles,
        "warning": (
            "Divergência preservada para revisão posterior; nenhum perfil foi renomeado, "
            "criado ou mapeado automaticamente."
        ) if missing_profiles else None,
    }

def verificar_pronto_para_confirmacao():
    estado = obter_estado()
    mostrar_estado()
    if estado.get("valid") and estado.get("complete"):
        try:
            diagnostico = validar_perfis_nodes_scenario()
            print("\nValidação estrutural nível 1: OK")
            print(json.dumps(diagnostico, indent=2, ensure_ascii=False))
            if diagnostico["profiles_without_device_definition"]:
                print("\nAVISO: divergência CSV/[devices] preservada para revisão; não foi corrigida automaticamente.")
            print("\nCenário estruturalmente válido e completo; pode ser confirmado.")
        except Exception as exc:
            print(f"\nCenário bloqueado pela validação nível 1: {type(exc).__name__}: {exc}")
    else:
        print("\nCenário ainda não está pronto para confirmação estrutural.")
    return estado

def confirmar_cenario():
    estado = obter_estado()
    if not estado.get("valid"):
        raise RuntimeError("O cenário não está válido segundo a validação estrutural.")
    if not estado.get("complete"):
        raise RuntimeError("O cenário ainda está incompleto; os campos obrigatórios não serão ignorados.")
    diagnostico = validar_perfis_nodes_scenario()
    if diagnostico["profiles_without_device_definition"]:
        print("AVISO — divergência CSV/[devices] mantida sem alterações:")
        print(json.dumps(diagnostico, indent=2, ensure_ascii=False))
    resposta = conversar("Confirmo o cenário.")
    print(json.dumps(resposta, indent=2, ensure_ascii=False, default=str)
          if isinstance(resposta, (dict, list)) else resposta)
    print("\nEstado após confirmação:")
    mostrar_estado()
    return obter_estado()

verificar_pronto_para_confirmacao()


**A confirmação é uma ação explícita.** Mantenha `CONFIRMAR_CENARIO = False` enquanto estiver revisando os valores; altere para `True` e execute a célula seguinte somente quando estiver pronto.


In [ ]:
# Ação explícita e desativada por padrão.
CONFIRMAR_CENARIO = False

if CONFIRMAR_CENARIO:
    confirmar_cenario()
else:
    print("Confirmação desativada. Revise o cenário; só então defina CONFIRMAR_CENARIO = False e execute esta célula.")


## 5. Gerar `scenario.toml`

A gravação exige que o cenário esteja válido, completo e confirmado. O `nodes.csv` já foi preparado antes da conversa e não será recompilado nesta etapa. Esta é a única etapa que grava `scenario.toml`.

A gravação também é bloqueada se o validador pós-LLM encontrar erros. O relatório aparece na saída da geração.


**A geração fica desativada por padrão.** Se `scenario.toml` já existir, ele não será substituído sem habilitar explicitamente a substituição.


## Validação estrutural local e validação do SDK

Primeiro ocorre a validação estrutural local; em seguida o SDK chama o motor e executa `PlanningScenario.validate()` antes da gravação.


In [ ]:
"""Validação offline e sem efeitos colaterais para projetos VibePlanner.

Não substitui PlanningScenario.validate() do network-service.
Não importa cisei_planning_sdk, não registra cenários e não calcula enlaces.
"""
from __future__ import annotations
import csv
import json
import math
import os
import re
import tomllib
from datetime import datetime, timezone
from pathlib import Path

REQUIRED_CSV = ('site_id', 'device_profile', 'kind', 'lat', 'lon', 'x', 'y', 'mount_height_m', 'connected', 'can_route', 'rank')


def validate_project(project: str | Path, *, toml_text: str | None = None) -> dict:
    project = Path(project).resolve(strict=True)
    errors, warnings = [], []
    info = {'project': project.name}
    scenario_file = project / 'scenario.toml'
    nodes_file = project / 'nodes.csv'
    if toml_text is None and (not scenario_file.is_file() or scenario_file.is_symlink()):
        errors.append('scenario.toml ausente ou link simbólico.')
    if not nodes_file.is_file() or nodes_file.is_symlink():
        errors.append('nodes.csv ausente ou link simbólico.')
    scenario = None
    if toml_text is not None or scenario_file.is_file() and not scenario_file.is_symlink():
        try:
            if toml_text is not None:
                scenario = tomllib.loads(toml_text)
            else:
                with scenario_file.open('rb') as f:
                    scenario = tomllib.load(f)
            if not isinstance(scenario, dict):
                errors.append('scenario.toml: raiz inválida.')
        except (OSError, ValueError, tomllib.TOMLDecodeError) as exc:
            errors.append(f'scenario.toml inválido: {exc}')
    if scenario is not None:
        required_sections = ('scenario','antennas','interfaces','devices','metrics','instances')
        for section in required_sections:
            if not isinstance(scenario.get(section), dict) or not scenario[section]:
                errors.append(f'Seção [{section}] ausente ou vazia.')
        crs = scenario.get('scenario', {}).get('working_crs') if isinstance(scenario.get('scenario'), dict) else None
        if not isinstance(crs, str) or not crs.strip():
            errors.append('scenario.working_crs obrigatório.')
        info['working_crs'] = crs
        inst = scenario.get('instances', {})
        if isinstance(inst, dict):
            for k, expected in (('source','nodes.csv'),('format','csv'),('profile_column','device_profile')):
                if inst.get(k) != expected:
                    errors.append(f'instances.{k} deve ser {expected!r} (recebido {inst.get(k)!r}).')
        ants = scenario.get('antennas', {})
        interfaces = scenario.get('interfaces', {})
        devices = scenario.get('devices', {})
        metrics = scenario.get('metrics', {})
        ants = ants if isinstance(ants, dict) else {}
        interfaces = interfaces if isinstance(interfaces, dict) else {}
        devices = devices if isinstance(devices, dict) else {}
        metrics = metrics if isinstance(metrics, dict) else {}
        for name, a in ants.items():
            if not isinstance(a, dict):
                errors.append(f'antennas.{name}: esperado objeto.'); continue
            kind = a.get('kind')
            if kind not in ('sector','omni'):
                errors.append(f'antennas.{name}.kind inválido: {kind!r}.')
            if kind == 'sector' and not _finite(a.get('beamwidth_deg'), positive=True):
                errors.append(f'antennas.{name}.beamwidth_deg deve ser positivo.')
            for k in ('height_m','azimuth_deg','downtilt_deg'):
                if k in a and not _finite(a[k]):
                    errors.append(f'antennas.{name}.{k} deve ser finito.')
        for name, iface in interfaces.items():
            if not isinstance(iface, dict):
                errors.append(f'interfaces.{name}: esperado objeto.'); continue
            if iface.get('antenna_id') not in ants:
                errors.append(f'interfaces.{name}: antenna_id inexistente: {iface.get("antenna_id")!r}.')
            if not _finite(iface.get('freq_mhz'), positive=True):
                errors.append(f'interfaces.{name}.freq_mhz deve ser positivo.')
            if not _finite(iface.get('tx_power_dbm')):
                errors.append(f'interfaces.{name}.tx_power_dbm deve ser finito.')
            tech = iface.get('tech')
            if not isinstance(tech, str) or tech not in metrics:
                errors.append(f'interfaces.{name}.tech sem metrics correspondente: {tech!r}.')
        for name, dev in devices.items():
            if not isinstance(dev, dict):
                errors.append(f'devices.{name}: esperado objeto.'); continue
            ids = dev.get('interfaces')
            if not isinstance(ids, list) or not ids:
                errors.append(f'devices.{name}.interfaces deve ser lista não vazia.')
            else:
                for iid in ids:
                    if not isinstance(iid, str) or iid not in interfaces:
                        errors.append(f'devices.{name}: interface inexistente: {iid!r}.')
        for name, m in metrics.items():
            if not isinstance(m, dict) or not isinstance(m.get('spec'), str) or not m['spec'].strip():
                errors.append(f'metrics.{name}.spec deve ser texto não vazio.')
    if nodes_file.is_file() and not nodes_file.is_symlink():
        try:
            with nodes_file.open('r', encoding='utf-8-sig', newline='') as f:
                reader = csv.DictReader(f)
                fields = reader.fieldnames or []
                missing = [k for k in REQUIRED_CSV if k not in fields]
                if missing:
                    errors.append('nodes.csv: colunas ausentes: ' + ', '.join(missing))
                ids = set()
                count = 0
                profiles = set()
                for rownum, row in enumerate(reader, 2):
                    count += 1
                    sid = (row.get('site_id') or '').strip()
                    profile = (row.get('device_profile') or '').strip()
                    if not sid: errors.append(f'nodes.csv linha {rownum}: site_id vazio.')
                    elif sid in ids: errors.append(f'nodes.csv linha {rownum}: site_id duplicado: {sid}.')
                    ids.add(sid)
                    if not profile: errors.append(f'nodes.csv linha {rownum}: device_profile vazio.')
                    profiles.add(profile)
                    for k, lo, hi in (('lat',-90,90),('lon',-180,180)):
                        value = _number(row.get(k))
                        if value is None or not lo <= value <= hi:
                            errors.append(f'nodes.csv linha {rownum}: {k} inválido.')
                    for k in ('x','y','mount_height_m'):
                        value = _number(row.get(k))
                        if value is None or (k == 'mount_height_m' and value < 0):
                            errors.append(f'nodes.csv linha {rownum}: {k} inválido.')
                    for k in ('connected','can_route'):
                        if (row.get(k) or '').strip().lower() not in ('true','false','1','0'):
                            errors.append(f'nodes.csv linha {rownum}: {k} não é booleano.')
                    rank = (row.get('rank') or '').strip()
                    if rank and _number(rank) is None:
                        errors.append(f'nodes.csv linha {rownum}: rank inválido.')
                info['nodes_count'] = count
                info['profiles_used'] = sorted(profiles)
                if count == 0: errors.append('nodes.csv sem instâncias.')
                if scenario is not None and isinstance(scenario.get('devices'), dict):
                    for profile in sorted(profiles - set(scenario['devices'])):
                        errors.append(f'Perfil usado no nodes.csv não definido em [devices]: {profile}.')
        except (OSError, UnicodeError, csv.Error) as exc:
            errors.append(f'Falha ao ler nodes.csv: {exc}')
    # A checagem do motor é intencionalmente não executada nesta versão.
    return {'timestamp': datetime.now(timezone.utc).isoformat(), 'project': project.name,
            'level1_ok': not errors, 'engine_validation': 'not_run',
            'planning_authorized': False, 'errors': errors, 'warnings': warnings,
            'info': info}


def _number(value):
    try:
        if isinstance(value, bool): return None
        n = float(value)
        return n if math.isfinite(n) else None
    except (TypeError, ValueError, OverflowError):
        return None


def _finite(value, positive=False):
    n = _number(value)
    return n is not None and (not positive or n > 0)



### Validação do SDK antes de gravar o TOML

Esta etapa chama `PlanningClient.define_scenario()` com um identificador exclusivo, obtém `validation_errors` produzidos por `PlanningScenario.validate()` e bloqueia a gravação quando houver erros. **A chamada registra um cenário temporário na memória do network-service**; não usa o ID do projeto ativo e não executa métricas, solve ou avaliação. O serviço deve estar acessível.


In [ ]:
# Validação real do SDK / network-service, com ID isolado.
# DefineScenario executa PlanningScenario.validate() no servidor.
import uuid
from pathlib import Path


def validar_cenario_com_sdk(toml_text: str) -> dict:
    try:
        from cisei_planning_sdk.client import PlanningClient
    except ImportError:
        from cisei_planning_sdk.client import PlannerClient as PlanningClient

    # Um ID exclusivo evita apagar o planejador/cache de um cenário ativo.
    validation_id = f"validation_{PROJECT_NAME}_{uuid.uuid4().hex}"
    temp_path = PROJECT_ROOT / f".sdk_validation_{uuid.uuid4().hex}.toml"
    try:
        temp_path.write_text(toml_text, encoding="utf-8")
        client = criar_cliente_validacao_sdk()
        associar_projeto_validacao(client, PROJECT_ROOT)
        # Este método chama POST /planning/scenarios/define; o serviço executa
        # PlanningScenario.validate() e retorna validation_errors.
        result = client.define_scenario(
            name=validation_id,
            scenario=temp_path,
            planner="cell",
        )
        errors = list(result.validation_errors or [])
        report = {
            "validator": "cisei_planning_sdk / PlanningScenario.validate()",
            "scenario_id": validation_id,
            "sdk_ok": bool(result.ok) and not errors,
            "validation_errors": errors,
            "summary": result.summary,
        }
        return report
    finally:
        temp_path.unlink(missing_ok=True)



In [ ]:
def gerar_e_salvar_scenario(overwrite=False):
    estado = obter_estado()
    for campo in ("valid", "complete", "confirmed"):
        if not estado.get(campo):
            raise RuntimeError(f"Não é possível gerar TOML: {campo} não está verdadeiro.")

    # Não sobrescrever silenciosamente um artefato preexistente.
    if SCENARIO_PATH.exists() and not overwrite:
        raise FileExistsError(
            f"{SCENARIO_PATH} já existe. Para substituí-lo, habilite "
            "SUBSTITUIR_SCENARIO_TOML explicitamente."
        )

    toml_text = agent.scenario.generate_toml()
    if not isinstance(toml_text, str) or not toml_text.strip():
        raise RuntimeError("ScenarioBuilder não retornou conteúdo TOML.")

    # Nível 1: interpretar novamente o texto antes de qualquer gravação.
    try:
        parsed_toml = tomllib.loads(toml_text)
    except (tomllib.TOMLDecodeError, ValueError) as exc:
        raise ValueError(f"scenario.toml inválido; gravação bloqueada: {exc}") from exc
    if not isinstance(parsed_toml, dict):
        raise ValueError("scenario.toml precisa conter uma tabela raiz.")
    instances_toml = parsed_toml.get("instances")
    if not isinstance(instances_toml, dict):
        raise ValueError("scenario.toml precisa conter a tabela [instances].")
    if instances_toml.get("source") != "nodes.csv":
        raise ValueError(
            "scenario.toml precisa referenciar instances.source = 'nodes.csv'; "
            f"valor encontrado: {instances_toml.get('source')!r}."
        )

    # Validação nível 1 no TOML efetivamente gerado, antes de qualquer gravação.
    diagnostico = validar_perfis_nodes_scenario(parsed_toml)
    if diagnostico["profiles_without_device_definition"]:
        raise ValueError(
            "Geração bloqueada: perfis de nodes.csv sem definição em [devices]: "
            + ", ".join(diagnostico["profiles_without_device_definition"])
        )
    if instances_toml.get("format") != "csv" or instances_toml.get("profile_column") != "device_profile":
        raise ValueError("Geração bloqueada: instances.format/profile_column incompatíveis com nodes.csv.")
    print("Validação estrutural nível 1: TOML e perfis compatíveis com nodes.csv.")
    print("Diagnóstico de perfis (somente aviso; divergências não são corrigidas automaticamente):")
    print(json.dumps(diagnostico, indent=2, ensure_ascii=False))

    # Revalidar nodes.csv na etapa final; não recompila nem modifica o original.
    NODES_COMPILER.read_csv(NODES_PATH)

    # Validação estrutural adicional: TOML gerado em memória + CSV real.
    # Não usa endpoints do serviço e não altera o planejamento em execução.
    report = validate_project(PROJECT_ROOT, toml_text=toml_text)
    print("=== RELATÓRIO DE VALIDAÇÃO PÓS-LLM (NÍVEL 1) ===")
    print(json.dumps(report, indent=2, ensure_ascii=False))
    if not report["level1_ok"]:
        raise ValueError(
            "Gravação bloqueada pela validação pós-LLM:\n- "
            + "\n- ".join(report["errors"])
        )
    # Validação obrigatória feita pelo SDK, antes de gravar scenario.toml.
    sdk_report = validar_cenario_com_sdk(toml_text)
    print("=== VALIDAÇÃO DO SDK / MOTOR ===")
    print(json.dumps(sdk_report, indent=2, ensure_ascii=False, default=str))
    if not sdk_report["sdk_ok"]:
        raise ValueError(
            "Gravação bloqueada por PlanningScenario.validate():\n- "
            + "\n- ".join(map(str, sdk_report["validation_errors"]))
        )
    print("SDK: cenário aceito na definição; factibilidade RF não verificada.")

    # Escrita atômica; nenhum backup/relatório adicional fica no projeto.
    temp_path = SCENARIO_PATH.with_name("scenario.toml.tmp")
    try:
        temp_path.write_text(toml_text, encoding="utf-8")
        temp_path.replace(SCENARIO_PATH)
    finally:
        if temp_path.exists():
            temp_path.unlink()

    print(f"scenario.toml salvo: {SCENARIO_PATH}")
    print(f"Tamanho: {SCENARIO_PATH.stat().st_size:,} bytes")
    return toml_text


**Ação manual:** habilite `GERAR_SCENARIO_TOML` somente depois da confirmação. Se o arquivo já existir, `SUBSTITUIR_SCENARIO_TOML` também precisa ser habilitado explicitamente.


In [ ]:
# Ações de gravação desativadas por padrão.
GERAR_SCENARIO_TOML = False
SUBSTITUIR_SCENARIO_TOML = False

if GERAR_SCENARIO_TOML:
    scenario_toml = gerar_e_salvar_scenario(overwrite=SUBSTITUIR_SCENARIO_TOML)
else:
    print("Geração desativada. Após confirmar o cenário, defina GERAR_SCENARIO_TOML = False e execute esta célula.")


## 6. Verificar os dois artefatos de autoria

Esta verificação lê apenas `nodes.csv` e, se já tiver sido gerado, `scenario.toml`. Não executa ações externas nem cria arquivos adicionais.


In [ ]:
# Diagnóstico final dos dois artefatos, sem criar arquivos auxiliares.
if NODES_PATH.is_file():
    NODES_FINAL_ROWS = NODES_COMPILER.read_csv(NODES_PATH)

artefatos = {
    "nodes.csv": {
        "path": str(NODES_PATH),
        "exists": NODES_PATH.is_file(),
        "bytes": NODES_PATH.stat().st_size if NODES_PATH.is_file() else None,
    },
    "scenario.toml": {
        "path": str(SCENARIO_PATH),
        "exists": SCENARIO_PATH.is_file(),
        "bytes": SCENARIO_PATH.stat().st_size if SCENARIO_PATH.is_file() else None,
    },
}

if SCENARIO_PATH.is_file():
    scenario_data = tomllib.loads(SCENARIO_PATH.read_text(encoding="utf-8"))
    instances_data = scenario_data.get("instances", {})
    if instances_data.get("source") != "nodes.csv":
        raise ValueError(
            "scenario.toml precisa referenciar instances.source = \"nodes.csv\". "
            f"Valor encontrado: {instances_data.get('source')!r}."
        )

print(json.dumps(artefatos, indent=2, ensure_ascii=False))
print("Escopo do notebook: nodes.csv e scenario.toml; nenhum planejamento ou visualização é executado.")


In [ ]:
import sys
import importlib.util
import importlib.metadata as metadata

print("===== AMBIENTE PYTHON =====")
print("Executável:", sys.executable)

print("\n===== PACOTE INSTALADO =====")
try:
    print("Versão:", metadata.version("cisei-scenario-sdk"))
except metadata.PackageNotFoundError:
    print("cisei-scenario-sdk não encontrado")

print("\n===== MÓDULOS =====")

for module in (
    "cisei_scenario_sdk",
    "cisei_scenario_sdk.scenario_session",
    "cisei_scenario_sdk.scenario_builder",
    "cisei_scenario_sdk.scenario_project",
):
    try:
        spec = importlib.util.find_spec(module)
        print(f"{module}: {spec.origin if spec else 'NÃO ENCONTRADO'}")
    except (ImportError, ModuleNotFoundError) as exc:
        print(f"{module}: ERRO — {exc}")